# Pipeline Completo — Orquestador

Ejecuta el pipeline completo de analisis de seguridad:
clone → sbom → grype → codeql → report

**Curso:** Ciberseguridad (ICC610) — 2026


## 1. ⚙️ Configuración e Imports

In [1]:
import json
import subprocess
import sys
import os
import shutil
import pandas as pd
from pathlib import Path
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt

# Detectar PROJECT_ROOT automaticamente (devcontainer, Codespaces, local)
_search = Path(os.getcwd())
while not (_search / "data" / "config.json").exists():
    if _search.parent == _search:
        _search = Path(os.getcwd())
        break
    _search = _search.parent
PROJECT_ROOT = _search

REPOS_DIR = PROJECT_ROOT / "data" / "repos"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
CONFIG_FILE = PROJECT_ROOT / "data" / "config.json"

REPOS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

print("=" * 60)
print("  PIPELINE COMPLETO DE ANALISIS DE SEGURIDAD")
print("=" * 60)
print(f"  Proyecto:   {PROJECT_ROOT}")
print(f"  Repos:      {REPOS_DIR}")
print(f"  Resultados: {RESULTS_DIR}")
print(f"  Config:     {CONFIG_FILE}")
print(f"  Fecha:      {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

print("\nVerificando herramientas instaladas...")
tools = {"syft": "syft version",
         "grype": "grype version", "codeql": "codeql version"}
tools_ok = True
for tool, cmd in tools.items():
    result = subprocess.run(cmd.split(), capture_output=True, text=True)
    version = result.stdout.strip().split(
        "\n")[0] if result.returncode == 0 else "NO ENCONTRADO"
    status = "OK" if result.returncode == 0 else "ERROR"
    print(f"  [{status}] {tool:10} -> {version}")
    if result.returncode != 0:
        tools_ok = False

if tools_ok:
    print("\nTodas las herramientas estan disponibles.")
else:
    print("\nAlgunas herramientas no estan disponibles. Algunos pasos podrian fallar.")

  PIPELINE COMPLETO DE ANALISIS DE SEGURIDAD
  Proyecto:   /workspaces/PROYECTO_CIBER
  Repos:      /workspaces/PROYECTO_CIBER/data/repos
  Resultados: /workspaces/PROYECTO_CIBER/data/results
  Config:     /workspaces/PROYECTO_CIBER/data/config.json
  Fecha:      2026-05-07 21:12:42

Verificando herramientas instaladas...
  [OK] syft       -> Application:   syft
  [OK] grype      -> Application:         grype
  [OK] codeql     -> CodeQL command-line toolchain release 2.25.1.

Todas las herramientas estan disponibles.


### 1.1 Configurar repositorios a analizar

Modifica la lista `REPOSITORIES` para cambiar qué repositorios analizar.  
El `config.json` se actualizará automáticamente.

In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIGURACIÓN: Repositorios a analizar                     ║
# ║  Este bloque lee la configuración actual. Para cambiarla,   ║
# ║  edita directamente el archivo data/config.json             ║
# ╚══════════════════════════════════════════════════════════════╝

if CONFIG_FILE.exists():
    with open(CONFIG_FILE, "r") as f:
        config = json.load(f)
    print(f"✅ Cargando configuración existente desde {CONFIG_FILE}")
else:
    print(
        f"⚠️ No se encontró {CONFIG_FILE}. Creando configuración por defecto...")
    config = {
        "repos_dir": "data/repos",
        "output_dir": "data/results",
        "repositories": [
            "https://github.com/huggingface/transformers",
            "https://github.com/huggingface/kernels",
            "https://github.com/huggingface/dataset-viewer",
            "https://github.com/huggingface/sentence-transformers"
        ],
        "organizations": [],
        "clone_options": {
            "max_inactive_days": 30,
            "skip_archived": True,
            "skip_forks": True,
            "max_repos": 50
        },
        "tools": {
            "syft": {"enabled": True, "output_format": "json"},
            "grype": {"enabled": True, "output_format": "json", "update_db": True},
            "codeql": {
                "enabled": True,
                "output_format": "json",
                "supported_languages": ["python", "javascript", "java", "cpp", "csharp"]
            }
        },
        "concurrency": {
            "max_workers": 4,
            "enabled": True
        }
    }
    with open(CONFIG_FILE, "w") as f:
        json.dump(config, f, indent=4)

REPOSITORIES = config.get("repositories", [])
ORGANIZATIONS = config.get("organizations", [])

print("\n�� Configuración actual:")
print(f"  Repositorios individuales: {len(REPOSITORIES)}")
for r in REPOSITORIES:
    print(f"    → {r}")
if ORGANIZATIONS:
    print(f"  Organizaciones: {len(ORGANIZATIONS)}")
    for o in ORGANIZATIONS:
        print(f"    → {o}")

✅ Cargando configuración existente desde /workspaces/PROYECTO_CIBER/data/config.json

�� Configuración actual:
  Repositorios individuales: 4
    → https://github.com/huggingface/transformers
    → https://github.com/huggingface/kernels
    → https://github.com/huggingface/dataset-viewer
    → https://github.com/huggingface/sentence-transformers


---

## 2. 📥 Clonar Repositorios

Descarga los repositorios configurados en `data/repos/`.

In [3]:
print("═" * 60)
print("  [1/5] 📥 CLONANDO REPOSITORIOS")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "clone"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    # Listar repos clonados
    repos = [d.name for d in REPOS_DIR.iterdir() if d.is_dir()]
    print(f"\n📂 Repositorios disponibles en data/repos/: {repos}")

════════════════════════════════════════════════════════════
  [1/5] 📥 CLONANDO REPOSITORIOS
════════════════════════════════════════════════════════════
═══════════════════════════════════════
  CLONADOR DE REPOSITORIOS
═══════════════════════════════════════

📋 4 repositorio(s) individual(es) configurado(s)

🔄 Clonando 4 repositorio(s) — modo paralelo (5 workers)...
  ↓ Clonando: transformers
  ↓ Clonando: kernels
  ↓ Clonando: dataset-viewer
  ↓ Clonando: sentence-transformers
  ✓ Clonado: kernels
  ✓ Clonado: dataset-viewer
  ✓ Clonado: sentence-transformers
  ✓ Clonado: transformers
                                                                        
                          Resumen de Clonación                          
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Repositorio           ┃ Estado    ┃ Ruta                             ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ kernels               │ ✓ Clonado │ da

---

## 3. 📦 Generar SBOMs (Syft)

Genera la lista de componentes y dependencias de cada repositorio usando **Syft**.

In [4]:
print("═" * 60)
print("  [2/5] 📦 GENERANDO SBOMs (SYFT)")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "sbom"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    sbom_files = sorted(RESULTS_DIR.glob("*-sbom.json"))
    print(f"\n✅ {len(sbom_files)} SBOM(s) generados")
    for f in sbom_files:
        size = f.stat().st_size / 1024
        print(f"   📄 {f.name} ({size:.1f} KB)")

════════════════════════════════════════════════════════════
  [2/5] 📦 GENERANDO SBOMs (SYFT)
════════════════════════════════════════════════════════════
═══════════════════════════════
GENERADOR DE SBOMs - Syft
═══════════════════════════════

📦 Procesando 4 repositorio(s) — modo paralelo (5 workers)...

Generando SBOM para: dataset-viewer

Generando SBOM para: kernels

Generando SBOM para: sentence-transformers

Generando SBOM para: transformers
✓ SBOM generado: data/results/kernels-sbom.json
✓ SBOM generado: data/results/sentence-transformers-sbom.json
✓ SBOM generado: data/results/dataset-viewer-sbom.json
✓ SBOM generado: data/results/transformers-sbom.json

Resumen guardado en: data/results/sbom-summary.json


✅ 4 SBOM(s) generados
   📄 dataset-viewer-sbom.json (5655.3 KB)
   📄 kernels-sbom.json (1078.4 KB)
   📄 sentence-transformers-sbom.json (27.1 KB)
   📄 transformers-sbom.json (306.9 KB)


---

## 4. 🔓 Escanear Vulnerabilidades (Grype)

Busca CVEs conocidos en las dependencias detectadas usando **Grype**.

> ℹ️ La primera ejecución descarga la base de datos de vulnerabilidades (~2 min).

In [5]:
print("═" * 60)
print("  [3/5] 🔓 ESCANEANDO VULNERABILIDADES (GRYPE)")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "grype"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT),
    timeout=600
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    grype_files = sorted(RESULTS_DIR.glob("*-grype.json"))
    print(f"\n✅ {len(grype_files)} escaneo(s) completados")
    for f in grype_files:
        with open(f) as fh:
            data = json.load(fh)
        n_vulns = len(data.get("matches", []))
        print(f"   🔓 {f.stem}: {n_vulns} vulnerabilidades")

════════════════════════════════════════════════════════════
  [3/5] 🔓 ESCANEANDO VULNERABILIDADES (GRYPE)
════════════════════════════════════════════════════════════
═══════════════════════════════
ESCANER DE VULNERABILIDADES - Grype
═══════════════════════════════

Actualizando base de datos de Grype...

🔓 Escaneando 4 repositorio(s) — modo paralelo (5 workers)...

Escaneando vulnerabilidades: dataset-viewer

Escaneando vulnerabilidades: kernels

Escaneando vulnerabilidades: sentence-transformers

Escaneando vulnerabilidades: transformers
  → usando SBOM: kernels-sbom.json
  → usando SBOM: dataset-viewer-sbom.json
  → usando SBOM: sentence-transformers-sbom.json
  → usando SBOM: transformers-sbom.json
✓ Escaneo completado: 0 vulnerabilidades encontradas
✓ Escaneo completado: 5 vulnerabilidades encontradas
✓ Escaneo completado: 14 vulnerabilidades encontradas
✓ Escaneo completado: 634 vulnerabilidades encontradas
                  Resumen de Escaneo                  
┏━━━━━━━━━━━━━━━

---

## 5. 🔍 Análisis Estático de Código (CodeQL)

Encuentra vulnerabilidades en el código fuente (SQL injection, XSS, etc.) usando **CodeQL**.

> ⏳ **Este paso puede tardar 5-20 minutos** por repositorio. Si deseas saltarlo, comenta la celda y continúa.

In [ ]:
print("═" * 60)
print("  [4/5] 🔍 ANÁLISIS ESTÁTICO (CODEQL)")
print("═" * 60)
print("⏳ Este paso puede tardar varios minutos...\n")

result = subprocess.run(
    ["uv", "run", "python", "main.py", "codeql"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT),
    timeout=1800  # 30 min max
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
else:
    codeql_files = sorted(RESULTS_DIR.glob("*-codeql.json"))
    print(f"\n✅ {len(codeql_files)} análisis completados")
    for f in codeql_files:
        with open(f) as fh:
            data = json.load(fh)
        n_findings = data.get("total", len(data.get("findings", [])))
        print(f"   🔍 {f.stem}: {n_findings} hallazgos")

════════════════════════════════════════════════════════════
  [4/5] 🔍 ANÁLISIS ESTÁTICO (CODEQL)
════════════════════════════════════════════════════════════
⏳ Este paso puede tardar varios minutos...



---

## 6. 📊 Generar Reporte Consolidado

In [ ]:
print("═" * 60)
print("  [5/5] 📊 GENERANDO REPORTE CONSOLIDADO")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "report"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    report_file = RESULTS_DIR / "consolidated-report.json"
    if report_file.exists():
        with open(report_file) as f:
            report = json.load(f)
        print("\n📋 Contenido del reporte:")
        print(json.dumps(report, indent=2))

print("\n" + "═" * 60)
print("  ✅ PIPELINE COMPLETO EJECUTADO")
print("═" * 60)

# Resumen de archivos generados
print("\n📂 Archivos generados en data/results/:")
for f in sorted(RESULTS_DIR.iterdir()):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f"   {'📄' if size < 100 else '📦'} {f.name:40} ({size:.1f} KB)")

════════════════════════════════════════════════════════════
  [5/5] 📊 GENERANDO REPORTE CONSOLIDADO
════════════════════════════════════════════════════════════
═══════════════════════════════
GENERADOR DE REPORTES
═══════════════════════════════
          SBOMs Generados          
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Repositorio           ┃ Estado  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ sentence-transformers │ success │
│ kernels               │ success │
│ dataset-viewer        │ success │
│ transformers          │ success │
└───────────────────────┴─────────┘
               Vulnerabilidades (Grype)               
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Repositorio           ┃ Vulnerabilidades ┃ Estado  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ sentence-transformers │ 0                │ success │
│ transformers          │ 5                │ success │
│ kernels               │ 12               │ success │
│ dataset-viewer        │ 619              